# Explore post-processed Steam data

Sanity-checks on the output of `preprocess_steam.ipynb` and `item_cf_steam.ipynb`: split sizes, id consistency, interaction distributions, product table, the simulator jsonl, and the item similarity matrix.

# 0. Import

In [1]:
import os
import json

import numpy as np
import pandas as pd
from local_package.config.data import STEAM_PROCESSED_DIR

# 1. Configuration

In [2]:
# --- config: point this at the same OUTPUT_DIR used in preprocess_steam.ipynb / item_cf_steam.ipynb ---
OUTPUT_DIR = STEAM_PROCESSED_DIR / "chatbot"

# 2. Load outputs

In [3]:
df_train = pd.read_csv(OUTPUT_DIR / "train.tsv")
df_valid = pd.read_csv(OUTPUT_DIR / "valid.tsv")
df_test = pd.read_csv(OUTPUT_DIR / "test.tsv")
user_history = pd.read_csv(OUTPUT_DIR / "user_history.tsv")
products = pd.read_feather(OUTPUT_DIR / "products.ftr")

ArrowInvalid: File is too small to be a well-formed file

In [ ]:
with open(OUTPUT_DIR / "map.json") as f:
    id_maps = json.load(f)

In [ ]:
simulator_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith("simulator_test_data_")]
simulator_path = OUTPUT_DIR / simulator_files[0]
simulator_data = [json.loads(line) for line in open(simulator_path)]

In [ ]:
item_sim = np.load(OUTPUT_DIR / "item_sim.npy")

In [ ]:
print("train:", df_train.shape)
print("valid:", df_valid.shape)
print("test:", df_test.shape)
print("user_history (train+valid):", user_history.shape)
print("products:", products.shape)
print("users in map:", len(id_maps["user"]))
print("items in map:", len(id_maps["item"]))
print("simulator records:", len(simulator_data))
print("item_sim shape:", item_sim.shape, "dtype:", item_sim.dtype)

## Sanity checks

In [ ]:
# leave-one-out: every user should appear exactly once in valid and once in test
valid_counts = df_valid["user_id"].value_counts()
test_counts = df_test["user_id"].value_counts()
print("users with != 1 valid row:", (valid_counts != 1).sum())
print("users with != 1 test row:", (test_counts != 1).sum())

In [ ]:
# item ids in the splits should all exist in the product table
known_ids = set(products["id"])
for name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    missing = (~df["item_id"].isin(known_ids)).sum()
    print(f"{name}: {missing} item_ids missing from products table")

In [ ]:
# id range should match the map sizes
print("max user_id in train:", df_train["user_id"].max(), "vs users in map:", len(id_maps["user"]))
print("max item_id in train:", df_train["item_id"].max(), "vs items in map:", len(id_maps["item"]))

## Interaction distributions

In [ ]:
all_inter = pd.concat([df_train, df_valid, df_test])
print("interactions per user:\n", all_inter.groupby("user_id").size().describe())
print("\ninteractions per item:\n", all_inter.groupby("item_id").size().describe())

## Product table

In [ ]:
print("visited_num stats:")
print(products["visited_num"].describe())
print("\nmost-visited games:")
print(products.sort_values("visited_num", ascending=False)[["title", "visited_num"]].head(10))

In [ ]:
print("category distribution:")
print(products["category"].value_counts().head(20))
print(f"\nmissing/placeholder description: {(products['description'] == 'No description').mean():.1%}")

## Simulator jsonl peek

In [ ]:
for rec in simulator_data[:5]:
    print("history:", rec["history"][:200])
    print("target: ", rec["target"])
    print()

## Item similarity matrix (ItemCF)

Sanity-checks specific to `item_cf_steam.ipynb`'s output.

In [ ]:
n_items = len(id_maps["item"])
print("item_sim shape vs expected (n_items+1, n_items+1):", item_sim.shape, (n_items + 1, n_items + 1))

In [ ]:
# should be symmetric by construction (see build_cooccurrence_matrix in item_cf_steam.ipynb)
asymmetry = np.abs(item_sim - item_sim.T).max()
print("max |sim[i,j] - sim[j,i]| (expect ~0):", asymmetry)

In [ ]:
products_indexed = products.set_index("id")
sample_id = int(products.sample(1, random_state=2024)["id"].iloc[0])
top_k = np.argsort(-item_sim[sample_id])[1:6]  # skip index 0 (self-similarity slot / padding)
print("sample item:", products_indexed.loc[sample_id].title)
print("most similar items:")
for i in top_k:
    if i in products_indexed.index:
        print(" -", products_indexed.loc[i].title, f"(sim={item_sim[sample_id, i]:.3f})")